# 01 — Problem Framing

**Fase 1 — CRISP-DM: Business Understanding**

Este notebook enmarca el problema clínico y de negocio antes de tocar los datos, según `PRD_liver_disease_fases_0-3.md` (Fase 1). No se ejecuta ningún modelo ni se calculan métricas de clasificación — el objetivo es dejar el contexto, los objetivos y las preguntas de investigación explícitos por escrito.

## 1. Problema de negocio

Un grupo hospitalario busca **reducir la mortalidad por cirrosis hepática mejorando el diagnóstico temprano**. La cirrosis es fibrosis progresiva e irreversible del hígado, asociada a consumo de alcohol, hepatitis crónica y obesidad (MASLD).

El punto crítico es que **el hígado no duele**: no tiene nervios de dolor interno. Un paciente puede perder gran parte de su función hepática sin síntomas, y cuando aparecen signos visibles (ictericia, ascitis) el daño suele ser ya irreversible.

**Propuesta de valor:** un análisis de sangre barato (*Liver Function Test*, LFT) alimenta un modelo que actúa como **herramienta de cribado** (*screening tool*) y **sistema de apoyo a la decisión clínica** (*CDSS*). El modelo **no diagnostica**: prioriza a qué pacientes derivar al especialista, detectando el patrón bioquímico subclínico antes de que el cuerpo dé síntomas.

### 1.1 Ángulo diferenciador: auditoría de sesgo

Este proyecto no es un EDA genérico. Replica el marco de **Straw, I. & Wu, H. (2022)**, *Investigating for bias in healthcare algorithms: a sex-stratified analysis of supervised machine learning models in liver disease prediction* (BMJ Health Care Inform, 29(1)), que demostró **sesgo de sexo** sobre este mismo dataset: en los cuatro clasificadores evaluados por los autores, las mujeres presentan una **tasa de falsos negativos (FNR) más alta** — es decir, más diagnósticos perdidos.

Por esa razón, **todo el EDA y el preprocesamiento (Fases 2 y 3) se hacen adicionalmente estratificados por `Gender`**, además del análisis agregado que exige el enunciado académico.

## 2. *Label bias*: el concepto que sostiene la interpretación

La variable objetivo `Selector` **no es la verdad biológica**: es el diagnóstico emitido por un especialista dentro de un sistema clínico que ya tiene sesgos documentados (por ejemplo, menor sospecha de abuso de alcohol en mujeres, o umbrales de laboratorio de referencia definidos como unisex cuando lo "normal" difiere por sexo). El modelo no aprende *"quién tiene enfermedad hepática"* sino ***"a quién le diagnosticaron enfermedad hepática"***. Es un ***proxy label***.

**Consecuencia operativa para las Fases 2–3:** cada decisión de preprocesamiento (imputar por media, tratar outliers, escalar) puede **amplificar el sesgo hacia la minoría** (mujeres, ~24% de la muestra). Documentar ese riesgo en cada decisión técnica es lo que separa un informe mecánico de uno crítico — y es el eje que conecta este proyecto con el artículo de referencia.

## 3. Objetivos de ciencia de datos (Fases 0–3, medibles)

| ID | Objetivo | Métrica de cumplimiento |
|----|----------|--------------------------|
| OB-1 | Caracterizar completamente la estructura y calidad del dataset | Los problemas de calidad detectados (nulos, duplicados, desbalances, codificación del target) cuantificados y documentados |
| OB-2 | Cuantificar la forma de las distribuciones de biomarcadores | *Skewness* y *kurtosis* reportados para las 9 variables numéricas |
| OB-3 | Identificar redundancia informativa entre predictores | Todos los pares con \|r\| > 0.7 listados con decisión de retención |
| OB-4 | Entregar un dataset limpio, imputado y escalado, reproducible desde cero | `03_preprocessing.ipynb` corre *restart & run all* sin error y escribe `data/processed/` |
| OB-5 | Documentar el riesgo diferencial de sesgo por sexo introducido por el preprocesamiento | Sección dedicada en el informe, con evidencia estratificada |

Estos objetivos son el techo de la Actividad 1 académica y el piso del proyecto de portafolio: cubrirlos al 100% es obligatorio; documentarlos con evidencia estratificada por sexo es lo que eleva el análisis.

## 4. Preguntas de investigación

1. ¿Qué variables de laboratorio distinguen mejor a los pacientes hepáticos?
2. ¿Existen diferencias en las distribuciones de biomarcadores entre hombres y mujeres?
3. ¿El desbalance de sexo observado en la muestra puede inducir sesgo en un modelo futuro?
4. ¿Qué variables presentan mayor multicolinealidad y cómo afecta eso a la selección de características?
5. ¿Qué decisiones de preprocesamiento tienen el potencial de amplificar el sesgo hacia la minoría (mujeres)?

> 🔁 **Nota de iteración (Loop A, CRISP-DM):** estas preguntas son el punto de partida y se revisarán después de la Fase 2 (EDA). Es esperado — y correcto, dentro de la metodología iterativa— que un hallazgo del EDA (por ejemplo, la magnitud real del desbalance de sexo) obligue a reformular o priorizar alguna de estas preguntas. Cualquier cambio se registrará en `docs/CHANGELOG_iteraciones.md` con fecha, disparador y decisión tomada.

## 5. Criterios de éxito

| Tipo | Criterio |
|------|----------|
| Académico | Las 8 tareas del enunciado (T1–T8) respondidas **e interpretadas**; informe ≤ 10 páginas; notebook anexo ejecutable |
| Técnico | Reproducibilidad total: `git clone` → `pip install -r requirements.txt` → *restart & run all* → mismos resultados |
| Portafolio | README que explique el problema, el hallazgo de sesgo y cómo reproducirlo, legible por un reclutador en 3 minutos |

**Regla de prioridad:** el valor agregado nunca sustituye al requisito académico. Cuando el enunciado exige una técnica inferior a la mejor práctica disponible (por ejemplo, imputación por media en vez de una fórmula determinista), se implementan **ambas** y se comparan — nunca se reemplaza lo exigido por la alternativa superior.

## 6. Supuestos y limitaciones

### 6.1 Ausencia de factores de riesgo causales (consumo de alcohol, hepatitis)

El dataset **no contiene** variables sobre consumo de alcohol ni diagnóstico de hepatitis viral, pese a que estos son dos de los factores de riesgo causales más relevantes para la cirrosis mencionados en el contexto clínico del problema. Esta es una **limitación estructural del dataset**, no un error de procesamiento: el modelo únicamente puede aprender la **firma bioquímica** del daño hepático (bilirrubinas, enzimas, proteínas) a partir de un *Liver Function Test*, nunca su **etiología**.

Esta ausencia **refuerza** el argumento de *label bias* de la Sección 2: si no sabemos por qué un paciente desarrolló la enfermedad, tampoco podemos verificar si el proceso diagnóstico que generó `Selector` fue igual de riguroso para todos los subgrupos de pacientes (por ejemplo, mujeres con enfermedad hepática por alcohol, que según la literatura tienden a ser menos sospechadas y diagnosticadas más tarde).

### 6.2 *Label bias* de la variable objetivo

Como se explicó en la Sección 2, `Selector` es un diagnóstico clínico, no una verdad biológica verificada independientemente. Cualquier patrón que el modelo aprenda reproduce (y potencialmente amplifica) los sesgos del proceso diagnóstico original.

### 6.3 Alcance de estas fases

Este PRD cubre únicamente las Fases 0–3 (CRISP-DM: Business + Data Understanding + Data Preparation). **No se entrena ningún modelo, no se calculan métricas de clasificación ni se hace *train/test split*** en estas fases — esas actividades quedan para la Actividad 2 (Fases 4–5) y para el desarrollo de portafolio (Fases 6–7), fuera del alcance de este documento.